# Quantum Fourier Transform (QFT)
Quantum Fourier Transform (QFT) serves as the backbone for Quantum Machine Learning, we focus on its role as a change of basis between two fundamental conjugate representations: Position and Momentum.

## Introduction
### A. What is QML used for?
QML is primarily used to find patterns in high-dimensional data where classical algorithms struggle with the "curse of dimensionality."
* Classification: Identifying species (like the Iris dataset) or medical anomalies.
* Generative Modeling: Creating new molecular structures for drug discovery.
* Optimization: Solving complex logistics or financial portfolio problems.
* Quantum Kernels: Using quantum physics to transform data into a space where it is more easily separable.
### B. The Theory: Qubits and Hilbert Space
In classical computing, a bit is 0 or 1. In QML, a qubit is a vector in a 2D complex space called Hilbert Space.

### The Basic Mathematics
A quantum state $|\psi\rangle$ is a linear combination (superposition) of basis states $|0\rangle$ and $|1\rangle$:$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$
* $\alpha$ and $\beta$ are complex amplitudes.
* The probability of measuring the state as $|0\rangle$ is $|\alpha|^2$.
* The probability of measuring the state as $|1\rangle$ is $|\beta|^2$.
* Normalization: $|\alpha|^2 + |\beta|^2 = 1$ (The total probability must be 100%).

### C. Position vs. Momentum: The Conjugate Basis
This is the most critical physical concept for students to understand before moving into the Quantum Fourier Transform (QFT). In quantum mechanics, certain variables come in "pairs" called conjugate variables.

### The Position Basis ($|x\rangle$)
Think of this as the "Computational Basis" ($|0\rangle, |1\rangle, |2\rangle \dots$). The information is localized. If a qubit is in state $|0\rangle$, we know exactly "where" it is.

### The Momentum Basis ($|p\rangle$)
This is the "Fourier Basis" ($|+\rangle, |-\rangle \dots$). The information is delocalized across the phase of the system. In this basis, the information is not at a specific point; it is encoded in the frequency or "rhythm" of the quantum state.

### The Heisenberg Link
The more precisely we know the position ($|x\rangle$), the less we know about the momentum ($|p\rangle$), and vice versa.
* Position: Information is in the amplitude (which qubit is on).
* Momentum: Information is in the phase (the relative angle between qubits).

### D. QML Logic: From Features to Phases
The magic of QML happens when we take classical data (like "Petal Length") and treat it as a Position. We then apply a quantum operation to shift that information into the Momentum (the phase).
* Classical Input ($x$): A localized number.
* Quantum Encoding: We put that number into a rotation gate, like $R_y(x)$.
* Phase Interference: By moving the data into the "Momentum" basis, we allow different data points to interfere with each other—just like waves in an ocean.

When we measure the system, we are looking at the result of that interference. If the waves add up (constructive interference), the "momentum" tells us the data belongs to Class A. If they cancel out (destructive interference), it belongs to Class B.

## 1. QFT as a Basis Change: Position vs. Momentum
In quantum mechanics, a particle's state can be described in the Position Basis $|x\rangle$ or the Momentum Basis $|p\rangle$. These are conjugate variables related by the Fourier Transform.
* Position Basis ($|x\rangle$): The information is "localized." In a computational circuit, this corresponds to the standard basis ($|0\rangle, |1\rangle, \dots$).
* Momentum Basis ($|p\rangle$): The information is "delocalized" across the phase of the system.

The QFT is the unitary operator $\mathcal{F}$ that rotates a state from the position basis to the momentum basis:$$|p\rangle = \mathcal{F} |x\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i x k / N} |k\rangle$$Why this matters for QML: Classical data is typically "localized" (e.g., a pixel value or a measurement). By applying QFT, we shift that information into the relative phases of a superposition. This is the quantum equivalent of moving from a spatial domain to a frequency domain, where certain patterns (periodicity, correlations) become mathematically "visible" to the quantum gates.

## 2. Deriving the QML System from QFT
To build a QML system using this physical intuition, we follow a derivation that treats the QFT as a Feature Map.
### Step A: Encoding (The Position Input)
We start by encoding our classical features $x$ into the computational basis. If we have a vector $x$, we prepare the state:$$|\psi_{pos}\rangle = |x\rangle$$
### Step B: The Feature Map (The Momentum Shift)
We apply the QFT to move into the momentum basis.$$|\psi_{mom}\rangle = QFT |x\rangle$$
Now, the information $x$ is encoded in the phase of every basis state. This represents a non-linear mapping into a high-dimensional Hilbert space.
### Step C: The Variational Ansatz (Learning in Momentum Space)
Instead of training on raw pixels, our QML model (the Ansatz $U(\theta)$) operates on the "frequencies" of the data.$$|\psi_{final}\rangle = U(\theta) |\psi_{mom}\rangle$$

In momentum space, simple rotations ($R_z$) can manipulate the periodic patterns within the data far more efficiently than they could in the position (raw) space.
### Step D: Measurement (The Observable)
To get a prediction, we measure the expectation value. Because we are in the momentum basis, measuring the Pauli-Z observable effectively probes the interference patterns created by the data.$$y = \langle \psi_{final} | Z | \psi_{final} \rangle$$
## 3. Physical Implementation (AWS Braket)
This code demonstrates how to use the QFT to "phase-encode" a simple piece of data, effectively preparing it for a QML classifier.

In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

In [ ]:
import numpy as np
from braket.circuits import Circuit, Observable
from braket.devices import LocalSimulator

def qft_feature_map(n_qubits):
    qc = Circuit()
    # 1. Start with a localized state (Position)
    # For iris, this would be an Ry encoding

    # 2. Apply QFT to move to Momentum Basis
    for i in range(n_qubits):
        qc.h(i)
        for j in range(i + 1, n_qubits):
            angle = 2 * np.pi / (2**(j - i + 1))
            qc.cphaseshift(j, i, angle)

    # Standard QFT includes a SWAP at the end
    for i in range(n_qubits // 2):
        qc.swap(i, n_qubits - i - 1)

    return qc

# Example: 3-qubit QFT Feature Map
n = 3
device = LocalSimulator()
circuit = qft_feature_map(n)

# We measure the momentum-space expectation
circuit.expectation(Observable.Z(), target=0)

result = device.run(circuit, shots=0).result()
print(f"Expectation value in Momentum Space: {result.values[0]}")

# Applying QFT to Problems
When we move from simple quantum circuits to Systems of Systems (SoS)—such as large-scale power grids, sensor networks, or industrial IoT—the Quantum Fourier Transform (QFT) stops being just a mathematical trick and becomes a physical "correlation engine."

In these complex environments, anomaly and fault detection are difficult because the signals are noisy, high-dimensional, and non-linearly coupled. Here is how we apply QFT to solve these challenges using the physics of position and momentum.

## 1. The SoS Mapping: Data as "Position"
In a System of Systems (e.g., a photovoltaic farm or a fleet of autonomous vehicles), each sensor provides a stream of data. We map these physical "locations" or sensor IDs to the Position Basis ($|x\rangle$) of our quantum system.
* State Preparation: We encode the status of $N$ sensors into $n = \log_2(N)$ qubits.
* The Global State: The quantum state $|\Psi\rangle = \sum c_i |i\rangle$ represents the simultaneous "snapshot" of the entire system.

## 2. QFT for Anomaly and Fault Detection
Anomaly detection is essentially the search for phase incoherence.

When a system is operating normally (in a "steady state"), the signals often have a periodic or predictable relationship. When a fault occurs (e.g., a short circuit in a PV string or a sensor failure), it introduces a "jump" or a "discontinuity" in the data.

### Moving to Momentum Space ($|p\rangle$)
By applying the QFT across the qubits representing the SoS, we shift the data from the sensor-specific position basis to the Momentum (Frequency) Basis.
* Nominal State: In momentum space, a healthy system displays constructive interference at specific "resonant frequencies." The momentum vector points in a predictable direction.
* Anomaly/Fault: A fault acts as a phase perturbation. This causes destructive interference in the momentum basis. The "energy" of the quantum state spreads out across the spectrum, making the anomaly immediately visible as a drop in the expectation value of the primary resonant frequency.

## 3. Object Detection: The "Quantum Signature"
In object detection (e.g., identifying a specific target in SAR or microwave imagery), we use the QFT to extract global features rather than local pixels.
* Feature Mapping: We encode image segments into quantum amplitudes.
* Frequency Analysis: Applying QFT reveals the "periodicity" of the object's shape. Natural objects (clutter) and man-made objects (targets) have distinct "momentum signatures.
* "Classification: A Variational Ansatz then "filters" these momentum signatures. Because the QFT has already delocalized the information, the Ansatz can recognize the object regardless of its specific "position" in the original data—this provides a degree of translational invariance.

## 4. Mathematical Derivation for Fault Detection
Let $x$ be a vector of sensor readings. We prepare $|\psi_x\rangle$. If a fault occurs at sensor $k$, the reading $x_k$ changes by $\Delta$.

The Momentum Shift:

Applying the QFT gives:$$|\psi_p\rangle = \mathcal{F}|\psi_x\rangle = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1} \left( \sum_{k=0}^{N-1} x_k e^{2\pi i j k / N} \right) |j\rangle$$

The Detection Logic:

In a healthy system, the inner product $\langle \psi_{nominal} | \psi_{current} \rangle \approx 1$.When a fault occurs, the term $x_k + \Delta$ creates a phase shift in the summation. Because the QFT spreads this $\Delta$ across all momentum basis states $|j\rangle$, the "Momentum Signature" shifts globally.We detect this by measuring the Expectation Value of a system-wide observable:$$\text{Anomaly Score} = 1 - \langle \Psi | \hat{M} | \Psi \rangle$$Where $\hat{M}$ is an operator tuned to the "healthy" momentum of the system.

## 5. Implementation Strategy (AWS Braket)
For a System of Systems, we utilize the n-qubit QFT.

In [ ]:
from braket.circuits import Circuit, Observable
import numpy as np

def sos_anomaly_detector(sensor_data):
    n_qubits = int(np.log2(len(sensor_data)))
    qc = Circuit()

    # 1. Encode SoS Data (Position)
    for i in range(n_qubits):
        qc.ry(i, sensor_data[i])

    # 2. QFT (Shift to Momentum)
    # [Standard QFT implementation here]

    # 3. Measurement (Momentum Signature)
    # Measure joint Z to look for global incoherence
    qc.expectation(Observable.Z(), target=0)

    return qc